#                                           Y2AI Project 2025 Water Pollution Disease





## Zero Waste
- Arad Soltanian - X00222268 - X00222268@myTUDublin.ie


## Summary Of Project Proposal


The project uses real data to study how water pollution drives disease risk, identifies vulnerable communities, and builds an AI model to predict health impacts showing how data can help make water safer.

## Dataset Details

- **Dataset : Water Pollution and Disease**
-  **Rows : 3,000**
-  **Features : 24**
- **License : MIT**

*Features:*
- *-Country*
- *-Region*
- *-Year*
- *-Water source type*
- *-Contaminant*
- *-Disease cases*

# 0. Setup

This section imports all required libraries for data handling, preprocessing, model training, evaluation, and saving.

In [3]:
# -------------------------------------------
# Import Libraries
# -------------------------------------------


import numpy as np # For Numerical operations
import pandas as pd # Data loading, cleaning, and manipulation

# Splitting data and scaling features
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Machine Learning models we tested
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# Metrics to check how well the models perform
from sklearn.metrics import (accuracy_score, f1_score, classification_report, mean_squared_error, r2_score, confusion_matrix)
import joblib

## 1. Data Cleaning & Wrangling

We loaded the dataset and performed basic inspection using .head(), .info(), and .describe() to understand its structure. We removed missing values and duplicate rows to ensure data quality. Column names were standardised by converting them to lowercase and replacing spaces with underscores for consistency and easier access in code. This step ensures the dataset is clean, consistent, and ready for feature selection and modeling.


In [4]:
df = pd.read_csv("water_pollution_disease.csv") # load dataset
df.head()                                       # Display first row
df.info()                                       # Show column types and null counts
df.describe()                                   # Statisical Summary

# Removing  missing values and duplicates
df = df.dropna()
df = df.drop_duplicates()

# Cleaning column names
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 24 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   Country                                        3000 non-null   object 
 1   Region                                         3000 non-null   object 
 2   Year                                           3000 non-null   int64  
 3   Water Source Type                              3000 non-null   object 
 4   Contaminant Level (ppm)                        3000 non-null   float64
 5   pH Level                                       3000 non-null   float64
 6   Turbidity (NTU)                                3000 non-null   float64
 7   Dissolved Oxygen (mg/L)                        3000 non-null   float64
 8   Nitrate Level (mg/L)                           3000 non-null   float64
 9   Lead Concentration (µg/L)                      3000 

,country,region,year,water_source_type,contaminant_level_(ppm),ph_level,turbidity_(ntu),dissolved_oxygen_(mg/l),nitrate_level_(mg/l),lead_concentration_(µg/l),...,"cholera_cases_per_100,000_people","typhoid_cases_per_100,000_people","infant_mortality_rate_(per_1,000_live_births)",gdp_per_capita_(usd),healthcare_access_index_(0-100),urbanization_rate_(%),sanitation_coverage_(%_of_population),rainfall_(mm_per_year),temperature_(°c),population_density_(people_per_km²)
0,Mexico,North,2015,Lake,6.06,7.12,3.93,4.28,8.28,7.89,...,33,44,76.16,57057,96.92,84.61,63.23,2800,4.94,593
1,Brazil,West,2017,Well,5.24,7.84,4.79,3.86,15.74,14.68,...,27,8,77.30,17220,84.73,73.37,29.12,1572,16.93,234
3,Nigeria,East,2016,Well,7.91,6.71,1.96,3.12,36.92,6.77,...,33,13,95.66,31166,39.07,71.07,94.25,937,3.79,555
4,Mexico,South,2005,Well,0.12,8.16,4.22,9.15,49.35,12.51,...,31,68,58.78,25661,23.03,55.55,69.23,2295,31.44,414
6,China,East,2022,River,0.06,6.11,3.12,6.97,44.98,1.23,...,23,90,92.59,6726,34.56,76.53,68.60,1573,22.42,584


# 2. Feature Selection and Pre-Processing

We selected our target variable: cholera_cases_per_100,000_people, which represents the number of cholera cases per 100,000 people. All other columns were treated as features.

To prepare the data:
- We dropped the target column from the feature set.
- Categorical variables were encoded using one-hot encoding to make them usable by machine learning models.
- The dataset was split into training and test sets (80/20 split) to evaluate model performance fairly.
- We applied standard scaling to the feature data to normalize values. This is especially important for models like Logistic Regression, which are sensitive to feature magnitude.

In [5]:
# set target and features
target = "cholera_cases_per_100,000_people"
y = df[target]
X = df.drop(columns=[target])

# Convert categorical coulumns into dummy encoded
X = pd.get_dummies(X, drop_first=True)

# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print shapes for debugging 
print("Data preprocessing complete.")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

# Show example row
print("\nSample input row:\n", X_train[0])
print("Corresponding target value:", y_train.iloc[0])

Data preprocessing complete.
X_train shape: (1802, 39)
X_test shape: (451, 39)
y_train shape: (1802,)
y_test shape: (451,)

Sample input row:
 [ 0.55437808  1.47329312  1.3956417   1.40883671  1.4575548  -0.94123421
 -1.1069551  -0.59798068 -1.37012093  0.73358566  1.53150426 -0.20265913
  0.61069565 -0.65105991  1.64174227  0.12274681  0.33959593 -0.97845522
  1.05830365 -0.33106783 -0.33722491 -0.34834483 -0.34129283 -0.33415486
 -0.31642076 -0.3237957  -0.32900013  2.96537995  1.97065188 -0.48573619
 -0.49444427 -0.5074463   2.26478513 -0.45937077 -0.46732302 -0.44602182
 -0.44064535 -0.69595205 -0.68981119]
Corresponding target value: 27


# 3. Model & Model Comparision 

We selected two models for comparison:

- **Logistic Regression**: A simple baseline model that is easy to interpret and performs well on linearly separable data.
- **Random Forest**: A more powerful ensemble method that builds multiple decision trees and combines their outputs. It handles complex relationships and feature interactions better.

Both models were trained on the same preprocessed training data. This allows us to compare their performance fairly using accuracy.


In [6]:
# Train Models, Logistic Regression and Random Forest
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
print("Models trained successfully.")

Models trained successfully.


# 4. Evaluation

We evaluated both models using accuracy and F1 score to measure overall performance and balance between precision and recall.

- **Accuracy** shows the percentage of correct predictions.
- **F1 Score** balances precision and recall, especially useful for imbalanced datasets.
- **Classification Report** provides a detailed breakdown of performance per class.

Random Forest achieved higher accuracy and F1 score compared to Logistic Regression, indicating it handled the complexity of the data better. However, we observed a warning due to one class having no predicted samples, which is common in imbalanced datasets. This does not affect overall functionality but highlights the importance of checking class distribution.

Based on these results, Random Forest is the stronger model for this task.


In [7]:

# Model Evaluation
y_pred_log = log_reg.predict(X_test)
y_pred_rf = rf.predict(X_test)

# Accuracy scores
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

# F1 scores
print("Logistic Regression F1:", f1_score(y_test, y_pred_log, average="weighted"))
print("Random Forest F1:", f1_score(y_test, y_pred_rf, average="weighted"))

# Full classification report for Random Forest
print("Random Forest Report:\n", classification_report(y_test, y_pred_rf))

Logistic Regression Accuracy: 0.008869179600886918
Random Forest Accuracy: 0.026607538802660754
Logistic Regression F1: 0.007936907881475509
Random Forest F1: 0.02233771145202785
Random Forest Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         9
           1       0.12      0.14      0.13         7
           2       0.00      0.00      0.00        10
           3       0.00      0.00      0.00        10
           4       0.00      0.00      0.00         8
           5       0.00      0.00      0.00         6
           6       0.00      0.00      0.00        11
           7       0.00      0.00      0.00        12
           8       0.00      0.00      0.00         8
           9       0.00      0.00      0.00         5
          10       0.05      0.10      0.07        10
          11       0.00      0.00      0.00        11
          12       0.00      0.00      0.00         7
          13       0.17      0.18      0.

c:\Users\plauz\Downloads\OneDrive_2025-12-04\Group Project Thursday Ida\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\plauz\Downloads\OneDrive_2025-12-04\Group Project Thursday Ida\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\plauz\Downloads\OneDrive_2025-12-04\Group Project Thursday Ida\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Us

# 5. Code that Saves Your Model for Demo

We saved the trained Random Forest model using 'joblib' so it can be reused in a demo or application. This allows us to load the model later without retraining.

In [8]:
# Save and load best model
joblib.dump(rf, "water_pollution_model.pkl")
model = joblib.load("water_pollution_model.pkl")

# 6. Demo Application 

We loaded the saved model and used it to make a prediction on a sample input from the test set. This demonstrates how the model can be used in a real-world scenario to estimate cholera risk based on input features.


In [9]:
# Sample prediction
sample = X_test[0].reshape(1,-1)
print("Predicted Risk:", model.predict(sample)[0])

Predicted Risk: 21


# 7. Ethics and Bias Reflection

**Data Source Validity**  
The dataset comes from public records on pollution and disease cases. We didn’t gather it ourselves, so we rely on how accurate and consistent the reporting is. Some countries may report better than others, which can affect the results.

**Bias Checks**  
There may be imbalance between countries or regions. Some places record more disease cases because they have better reporting systems. This could make the model think those regions are “higher risk” even when that might not be fully true.

**Limitations**  
The dataset doesn’t include other important factors like population, sanitation, weather, or healthcare quality. Our model can only use what is given. It can show patterns but cannot confirm that pollution directly caused the disease.

**Simplicity First**  
We start with simple models so we understand the results clearly. It also avoids overfitting and makes the project easier to explain.

**Privacy Considerations**  
There is no personal data, so privacy risks are low. Data is grouped by country/region only. We still treat it responsibly and avoid making strong claims about specific communities.

# 8. Conclusion
This notebook demonstrated the full machine learning workflow for predicting cholera risk from water pollution data. We cleaned and prepared the dataset, encoded categorical features, and applied scaling before training two models: Logistic Regression and Random Forest. Logistic Regression provided a simple baseline, while Random Forest achieved stronger performance and captured more complex patterns in the data. 
We evaluated both models using accuracy, F1 score, and classification reports, which highlighted the challenges of imbalanced data. Finally, we showed how the trained model can be saved and reloaded to make predictions on new samples, illustrating how this work could be extended into practical applications. Overall, the project highlights both the potential of machine learning in public health and the importance of careful preprocessing, evaluation, and ethical reflection.


